# Day 1 — Clean Superstore Sales Dataset

Module 4 · Part 4 — quy trình 3 bước dùng AI để inspect & clean data (áp dụng cho `Superstore.csv`).

## Step 1 — Đọc và mô tả tổng quan dữ liệu

In [ ]:
import pandas as pd

df = pd.read_csv("Superstore.csv", encoding="latin1")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.dtypes

Rows: 9994, Columns: 21


Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

**Nhận xét:** `Order Date` và `Ship Date` lẽ ra phải là kiểu ngày tháng (datetime) nhưng pandas đọc thành `object` (text) — cần kiểm tra sâu hơn ở Step 2.

## Step 2 — Tìm lỗi dữ liệu cụ thể

In [2]:
print("Completely duplicate rows:", df.duplicated().sum())
print("Duplicate Row IDs:", df.duplicated(subset=["Row ID"]).sum())
print("Missing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])

order_date_test = pd.to_datetime(df["Order Date"], format="%d-%m-%Y", errors="coerce")
ship_date_test = pd.to_datetime(df["Ship Date"], format="%d-%m-%Y", errors="coerce")
print("Order Date rows failing to parse:", order_date_test.isnull().sum())
print("Ship Date rows failing to parse:", ship_date_test.isnull().sum())

Completely duplicate rows: 0
Duplicate Row IDs: 0
Missing values per column:
 Series([], dtype: int64)
Order Date rows failing to parse: 0
Ship Date rows failing to parse: 0


**Nhận xét:** không có dòng trùng, không thiếu giá trị. Ngày tháng đang ở định dạng `DD-MM-YYYY` dạng text — cần convert sang datetime thật ở Step 3.

## Step 3 — Sửa lỗi và xác minh lại

In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d-%m-%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%d-%m-%Y")

print("Order Date dtype:", df["Order Date"].dtype)
print("Ship Date dtype:", df["Ship Date"].dtype)
print("Remaining null dates:", df["Order Date"].isnull().sum() + df["Ship Date"].isnull().sum())

Order Date dtype: datetime64[us]
Ship Date dtype: datetime64[us]
Remaining null dates: 0


## Step 4 — Lưu dữ liệu đã sạch ra file mới

In [ ]:
df.to_csv("Superstore-clean.csv", index=False)
print("Saved, checking:")
verify = pd.read_csv("Superstore-clean.csv", parse_dates=["Order Date", "Ship Date"])
print(verify.shape, verify["Order Date"].dtype)

Saved, checking:


(9994, 21) datetime64[us]


## Preview số liệu — chuẩn bị cho Day 2 (visualization)

In [5]:
print(df["Category"].value_counts())
print()
print(df["Region"].value_counts())
print()
print(df[["Sales", "Profit", "Discount", "Quantity"]].describe())

Category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

Region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64

              Sales       Profit     Discount     Quantity
count   9994.000000  9994.000000  9994.000000  9994.000000
mean     229.858001    28.656896     0.156203     3.789574
std      623.245101   234.260108     0.206452     2.225110
min        0.444000 -6599.978000     0.000000     1.000000
25%       17.280000     1.728750     0.000000     2.000000
50%       54.490000     8.666500     0.200000     3.000000
75%      209.940000    29.364000     0.200000     5.000000
max    22638.480000  8399.976000     0.800000    14.000000


## Ghi nhớ

- 3 Category: Furniture, Office Supplies, Technology.
- 4 Region: South, West, Central, East.
- `Profit` có thể âm (đơn hàng lỗ) — quan trọng khi phân tích Day 2/3.
- `Sales` dao động 0.44 – 22,638 USD, `Discount` 0 – 0.8.

➡️ Bước tiếp theo: mở file `CLAUDE.md` trong cùng thư mục để ghi lại data schema này.